# NB 62 — Custom-concept steering battery: multi-FRH vs subspace

Build concepts from **hand-provided word lists** (the Borda-ranked `core` field
of `resources/64_custom_concepts.json`) and steer with them across a battery of
single- and two-concept configurations, comparing the **multi-FRH** frame
pipeline (score-space voting) against the **subspace** pipeline, at two fluency
settings `λ ∈ {0, 0.4}`.

**Battery** (all from `core` words):

- **1-concept steering** (4 parts): `joy`, `sorrow`, `safe`, `harm`.
- **2-concept steering** (8 parts):
  - *repel* (attract A, repel B): `[safe,-harm]`, `[harm,-safe]`,
    `[joy,-sorrow]`, `[sorrow,-joy]`
  - *blend* (attract both): `[joy,safe]`, `[sorrow,safe]`, `[joy,harm]`,
    `[sorrow,harm]`

**Guide construction — both sides compose in *score space*, the fair twin.**
Multi-FRH routes every config through `generate_with_topk_multi_guide`, which
projects onto each concept per step, z-scores across the candidate pool, and
combines with weights: single = `guides=[A]` (`normalize=None`, which at `λ=0`
reproduces canonical single steering exactly); blend = `guides=[A,B],
weights=[1,1], normalize="zscore"`; repel = `weights=[1,-1]`. Subspace: single =
the basis; blend = `ConceptSubspace.blend([A,B])`; repel = a custom
difference-scored loop (`‖B_A·h‖ − ‖B_B·h‖`). So multi-FRH repel
(`z(proj_A) − z(proj_B)`) is the direct frame analogue of subspace repel.

**Fluency dial.** `generate_with_topk_multi_guide` and the subspace loops take
`fluency_weight = λ`, blending `cumulative concept score + λ · cumulative token
log-prob`. Both methods are swept over `λ ∈ {0, 0.4}` uniformly.

**Analysis.** Each run's steering effect is the **directed expression sum** over
its parts: `Σ sign · Δexpression`, where `Δ = expression(steered) −
expression(baseline)` measured against each *pure* concept, `sign = +1` for an
attracted concept, `−1` for a repelled one (single → `+ΔA`; blend → `+ΔA+ΔB`;
repel → `+ΔA−ΔB`). Results are the **sum over all parts** of each case
(1-concept, 2-concept), per method, per λ. Expression units differ across
methods (frame trace-correlation vs subspace ‖B·h‖), so those totals are
per-method; lexical-success rate and perplexity ratio are cross-comparable.

**Recipe:** bf16 Llama-3.1-8B, `k=4`, `steps=24`, `λ ∈ {0, 0.4}`,
`max_token_count=4`, subspace `rank=8`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from frames.evaluation.harness import EvaluationHarness
from frames.representations import FrameUnembeddingRepresentation
from subspace import ConceptSubspace, SubspaceUnembeddingRepresentation
from subspace.generation import (
    select_best_candidate,
    select_best_candidate_fluent,
    subspace_alignment,
)

In [ ]:
# -- parameters (papermill-overridable) -------------------------------------
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
CONCEPTS_JSON = "resources/64_custom_concepts.json"
WORD_FIELD = "core"  # which JSON field supplies the concept words

MAX_TOKENS, K, STEPS = 4, 4, 24
RANK = 8
LAMBDAS = [0.0, 0.4]  # fluency dial sweep (0 = pure steering)
SUFFIX = ""

# Battery — concept names must exist in the JSON.
SINGLES = [("joy",), ("sorrow",), ("safe",), ("harm",)]
REPELS = [("safe", "harm"), ("harm", "safe"), ("joy", "sorrow"), ("sorrow", "joy")]
BLENDS = [("joy", "safe"), ("sorrow", "safe"), ("joy", "harm"), ("sorrow", "harm")]

SMOKE = True  # subset prompts + `_smoke` outputs; flip to False for the full run

In [ ]:
# Neutral, open-ended prompts (chat turn + short assistant seed) — reused across
# both methods and every config. Steering pushes toward the concept regardless
# of topic, so the prompts are deliberately concept-agnostic.
QUESTIONS = [
    ("Tell me a short story.", "Once"),
    ("Describe your favorite place.", "My favorite place is"),
    ("What do you like to do on weekends?", "I like"),
    ("Describe the weather today.", "Today the weather is"),
    ("What should I cook for dinner?", "You should"),
    ("Write a paragraph about the city.", "The city"),
    ("Continue this thought.", "Here is what happened next:"),
    ("Describe a person you met.", "The person I met was"),
]


def chat(u, a=""):
    return (
        f"<|start_header_id|>user<|end_header_id|>{u}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>{a}"
    )


if SMOKE:
    QUESTIONS = QUESTIONS[:3]
    SUFFIX = SUFFIX or "_smoke"

PROMPTS = [chat(u, a) for u, a in QUESTIONS]
ENTRIES = {e["concept"]: e for e in json.loads(Path(CONCEPTS_JSON).read_text())}
WORDS = {name: ENTRIES[name][WORD_FIELD] for name in ENTRIES}
print(f"{len(PROMPTS)} prompts | concepts: {list(WORDS)} | field='{WORD_FIELD}'")

## Custom-concept builder

Turn a word list into the `(n, max_token_count)` padded token array that
`get_all_synsets` produces per synset row, mirroring the surface-form variations
and BOS-stripping of `frames/nlp/synsets.py`. Words whose tokenization exceeds
`max_token_count` are dropped (the same filter WordNet applies). Feeds
`compute_concept` unchanged on both pipelines.

In [ ]:
SURFACE_VARIATIONS = ("{term}", " {term}", "{term} ", " {term} ")


def custom_concept_dataframe(tokenizer, name, words, max_token_count):
    """One-row synset-style DataFrame ({synset, tokens}) from a word list."""
    pad_id, bos_id = tokenizer.pad_token_id, tokenizer.bos_token_id
    rows, kept = [], []
    for w in words:
        for v in SURFACE_VARIATIONS:
            ids = tokenizer(v.format(term=w.strip()), add_special_tokens=False)[
                "input_ids"
            ]
            if ids and ids[0] == bos_id:
                ids = ids[1:]
            if 0 < len(ids) <= max_token_count:
                rows.append(ids + [pad_id] * (max_token_count - len(ids)))
                kept.append(v.format(term=w.strip()))
    if not rows:
        raise ValueError(f"no variation of {name!r} fits max_token_count={max_token_count}")
    arr = np.array(rows, dtype=np.int64)
    return pd.DataFrame({"synset": [name], "tokens": [arr]}), kept


def word_token_len(tokenizer, word):
    """Token count of a plain word (no special tokens, BOS stripped)."""
    ids = tokenizer(word.strip(), add_special_tokens=False)["input_ids"]
    if ids and ids[0] == tokenizer.bos_token_id:
        ids = ids[1:]
    return len(ids)

## Load the model once, expose both representations

The frame representation loads the model; the subspace representation reuses the
same loaded model and unembedding (no second 8B load).

In [ ]:
fur = FrameUnembeddingRepresentation.from_model_id(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
sur = SubspaceUnembeddingRepresentation(
    model=fur.model,
    unembedding=fur,
    data=fur.data,
    rank=RANK,
    method="pooled",
)

LOG_PATH = Path(f"resources/64_battery{SUFFIX}.jsonl")
LOG_PATH.write_text("")  # fresh log per run
harness = EvaluationHarness(fur, LOG_PATH, min_lemmas_per_synset=1, max_token_count=MAX_TOKENS)

## Build concepts (both pipelines) + token-count report

In [ ]:
fconcept, sconcept, core_wordset = {}, {}, {}
tok = fur.model.tokenizer
tok_rows = []
for name, words in WORDS.items():
    df, kept = custom_concept_dataframe(tok, name, words, MAX_TOKENS)
    fconcept[name] = fur.compute_concept(df)   # frame Concept (1, d, k)
    sconcept[name] = sur.compute_concept(df)   # ConceptSubspace basis (d, rank)
    core_wordset[name] = {w.strip().lower() for w in words}

    lens = [word_token_len(tok, w) for w in words]
    n_fit = sum(length <= MAX_TOKENS for length in lens)
    tok_rows.append(
        {"concept": name, "n_words": len(words), "avg_tokens": np.mean(lens),
         "max_tokens": max(lens), "n_fit_<=%d" % MAX_TOKENS: n_fit,
         "frame_k": fconcept[name].tensor.shape[-1], "sub_rank": sconcept[name].rank}
    )

tok_df = pd.DataFrame(tok_rows)
overall_avg = np.mean([word_token_len(tok, w) for ws in WORDS.values() for w in ws])
print(tok_df.round(3).to_string(index=False))
print(f"\nAverage token count of core words across all four concepts: {overall_avg:.3f}")

## Configs, guide builders, and the subspace repel loop

Each config carries its `parts` — a list of `(concept, sign)` — that both the
guide construction and the directed-expression scoring read.

In [ ]:
def make_configs():
    cfgs = []
    for (a,) in SINGLES:
        cfgs.append({"name": a, "case": "1-concept", "kind": "single",
                     "parts": [(a, +1)]})
    for a, b in REPELS:
        cfgs.append({"name": f"{a}-{b}", "case": "2-concept", "kind": "repel",
                     "parts": [(a, +1), (b, -1)]})
    for a, b in BLENDS:
        cfgs.append({"name": f"{a}+{b}", "case": "2-concept", "kind": "blend",
                     "parts": [(a, +1), (b, +1)]})
    return cfgs


CONFIGS = make_configs()
print(f"{len(CONFIGS)} configs:")
for c in CONFIGS:
    print(f"  [{c['case']:9s}] {c['kind']:6s} {c['name']:14s} parts={c['parts']}")

In [ ]:
@torch.inference_mode()
def subspace_repel_generate(rep, input_text, pos, neg, k, steps, fluency_weight=0.0):
    """Subspace top-k generation: attract `pos`, repel `neg`.

    Mirrors ``SubspaceUnembeddingRepresentation.generate_with_topk_guide`` but
    scores each candidate by the DIFFERENCE of projection magnitudes
    ``‖B_pos·h‖ − ‖B_neg·h‖`` instead of a single alignment. With
    ``fluency_weight = λ > 0`` selection blends in cumulative token log-prob via
    :func:`select_best_candidate_fluent`, matching the native λ semantics.
    """
    input_text = [input_text] if isinstance(input_text, str) else input_text
    n = len(input_text)
    pos_basis = pos.basis.to(rep.model.device)
    neg_basis = neg.basis.to(rep.model.device)

    inputs = rep.model.make_input(input_text, padding=True)
    first, _, first_logprobs = rep._candidates(inputs["input_ids"], k)
    tokens = first.flatten(0, 1)
    cum_logprob = first_logprobs.flatten()

    probe = torch.zeros(n, 1)
    for _ in range(steps):
        sequences, last_hs, child_logprobs = rep._candidates(tokens, k)
        scores = subspace_alignment(last_hs, pos_basis) - subspace_alignment(
            last_hs, neg_basis
        )
        if fluency_weight:
            col_idx, probe = select_best_candidate_fluent(
                scores, cum_logprob, fluency_weight, n, k
            )
        else:
            col_idx, probe = select_best_candidate(scores, n, k)
        candidate_tokens = sequences.reshape(n, k, k, -1)
        tokens = candidate_tokens[np.arange(n), col_idx.cpu()].flatten(0, 1)
        if fluency_weight:
            rows = torch.arange(n, device=cum_logprob.device) * k + col_idx.to(
                cum_logprob.device
            )
            cum_logprob = (
                cum_logprob[rows].unsqueeze(-1) + child_logprobs[rows]
            ).flatten()
        if rep._is_generation_complete(tokens):
            break
    return rep.model.decode(tokens[::k]), probe.cpu()


def multifrh_steer(cfg, lam):
    """Score-space multi-guide steering (single / blend / repel) at fluency λ.

    All configs route through ``generate_with_topk_multi_guide``: each step
    projects onto every guide, z-scores across the candidate pool (for >1
    guide), and combines with weights (repel = weight −1). A single guide with
    ``normalize=None`` at λ=0 reproduces canonical single steering.
    """
    parts = cfg["parts"]
    guides = [fconcept[name] for name, _ in parts]
    weights = [float(sign) for _, sign in parts]
    normalize = None if len(guides) == 1 else "zscore"
    return fur.generate_with_topk_multi_guide(
        PROMPTS, guides=guides, weights=weights, k=K, steps=STEPS,
        normalize=normalize, fluency_weight=lam,
    )


def subspace_steer(cfg, lam):
    parts = cfg["parts"]
    if cfg["kind"] == "single":
        return sur.generate_with_topk_guide(
            PROMPTS, guide=sconcept[parts[0][0]], k=K, steps=STEPS, fluency_weight=lam
        )
    if cfg["kind"] == "blend":
        guide = ConceptSubspace.blend([sconcept[a] for a, _ in parts])
        return sur.generate_with_topk_guide(
            PROMPTS, guide=guide, k=K, steps=STEPS, fluency_weight=lam
        )
    (a, _), (b, _) = parts  # repel: custom difference-scored loop
    return subspace_repel_generate(
        sur, PROMPTS, sconcept[a], sconcept[b], K, STEPS, fluency_weight=lam
    )

## Baseline + directed-expression measurement helpers

In [ ]:
# Baseline (unsteered greedy) continuations — model-only, generate once.
baseline_texts = fur.generate(PROMPTS, max_new_tokens=STEPS + 2, do_sample=False)
baseline_conts = [harness.continuation(t, p) for t, p in zip(baseline_texts, PROMPTS)]


def frame_expr(text, name):
    return harness.concept_expression(text, fconcept[name])


def subspace_expr(text, name):
    if not text.strip():
        return 0.0
    with torch.inference_mode():
        return float(sur.expression(text, sconcept[name]).float().mean())


EXPR = {"multifrh": frame_expr, "subspace": subspace_expr}
STEER = {"multifrh": multifrh_steer, "subspace": subspace_steer}


def ppl_ratio(cont, base_cont):
    if not cont.strip() or not base_cont.strip():
        return float("nan")
    return harness.perplexity(cont) / harness.perplexity(base_cont)


def log(record):
    with LOG_PATH.open("a") as fh:
        fh.write(json.dumps(record) + "\n")

In [ ]:
for method in ("multifrh", "subspace"):
    expr = EXPR[method]
    for cfg in CONFIGS:
        attract = {w for name, sign in cfg["parts"] if sign > 0 for w in core_wordset[name]}
        for lam in LAMBDAS:
            texts, _probe = STEER[method](cfg, lam)
            for i, (prompt, text) in enumerate(zip(PROMPTS, texts)):
                cont = harness.continuation(text, prompt)
                base = baseline_conts[i]
                deltas, directed = {}, 0.0
                for name, sign in cfg["parts"]:
                    d = expr(cont, name) - expr(base, name)
                    deltas[name] = {"sign": sign, "delta": d}
                    directed += sign * d
                log({
                    "config": {"name": cfg["name"], "case": cfg["case"],
                               "kind": cfg["kind"], "method": method, "lam": lam},
                    "prompt": prompt,
                    "continuation": cont,
                    "deltas": deltas,
                    "directed_effect": directed,
                    "success": harness.contains_any(cont, attract),
                    "ppl_ratio": ppl_ratio(cont, base),
                })
            print(f"  {method:8s} {cfg['kind']:6s} {cfg['name']:14s} λ={lam} done")
print("battery complete ->", LOG_PATH)

## Results — sum of all parts, per case, per method, per λ

`directed_total` is the **sum** of the per-generation directed effects over every
part of the case (the headline the request asks for); `directed_mean` normalizes
it per generation. Directed effect is in each method's own expression units
(not cross-comparable); `success_rate` and `ppl_ratio` are cross-comparable.

In [ ]:
records = [json.loads(x) for x in LOG_PATH.read_text().strip().split("\n") if x]
rdf = pd.DataFrame(
    [{"method": r["config"]["method"], "case": r["config"]["case"],
      "kind": r["config"]["kind"], "name": r["config"]["name"],
      "lam": r["config"]["lam"], "directed": r["directed_effect"],
      "success": r["success"], "ppl_ratio": r["ppl_ratio"]} for r in records]
)

summary = (
    rdf.groupby(["method", "case", "lam"])
    .agg(directed_total=("directed", "sum"),
         directed_mean=("directed", "mean"),
         success_rate=("success", "mean"),
         ppl_ratio=("ppl_ratio", "mean"),
         n=("directed", "size"))
    .reset_index()
)
summary.to_csv(f"resources/64_battery_summary{SUFFIX}.csv", index=False)
print("=== sum of all parts, per case, per method, per λ ===")
print(summary.round(4).to_string(index=False))

# Secondary: per-kind breakdown within the 2-concept case.
print("\n=== 2-concept breakdown by kind ===")
brk = (
    rdf[rdf.case == "2-concept"].groupby(["method", "kind", "lam"])
    .agg(directed_total=("directed", "sum"), directed_mean=("directed", "mean"),
         success_rate=("success", "mean"), ppl_ratio=("ppl_ratio", "mean"),
         n=("directed", "size")).reset_index()
)
print(brk.round(4).to_string(index=False))

In [ ]:
CASES = ["1-concept", "2-concept"]
METHODS = ["multifrh", "subspace"]
colors = {"multifrh": "tab:green", "subspace": "tab:blue"}
panels = [("directed_total", "directed effect — sum of parts\n(per-method units)", None),
          ("success_rate", "attract-target lexical success", None),
          ("ppl_ratio", "perplexity ratio", 2.5)]
x = np.arange(len(CASES))
width = 0.38
fig, axes = plt.subplots(len(LAMBDAS), len(panels), figsize=(4 * len(panels), 3.4 * len(LAMBDAS)))
for row, lam in enumerate(LAMBDAS):
    for col, (metric, label, guard) in enumerate(panels):
        ax = axes[row, col]
        for j, m in enumerate(METHODS):
            vals = [summary[(summary.method == m) & (summary.case == c) & (summary.lam == lam)][metric].sum()
                    for c in CASES]
            ax.bar(x + (j - 0.5) * width, vals, width, label=m, color=colors[m])
        ax.set_xticks(x)
        ax.set_xticklabels(CASES)
        ax.set_title(f"{label} (λ={lam})")
        ax.axhline(0, color="gray", lw=0.6)
        if guard:
            ax.axhline(guard, color="red", ls=":", lw=0.8)
        ax.legend()
plt.suptitle("Custom-concept steering battery — multi-FRH vs subspace")
plt.tight_layout()
plt.savefig(f"resources/64_battery{SUFFIX}.png", dpi=150)
plt.show()

## Notes

- **Multi-FRH** composes in score space (z-scored per-concept projections,
  combined with weights) — the fair frame twin of the subspace approach, which
  also combines per-concept scores. Repel is `weights=[1,-1]`
  (`z(proj_A) − z(proj_B)`), mirroring the subspace `‖B_A·h‖ − ‖B_B·h‖`.
- **Directed effect** sums `sign · Δexpression` over a run's parts: single →
  `+ΔA`, blend → `+ΔA+ΔB`, repel → `+ΔA−ΔB`. Positive = intended direction.
  Measured against each *pure* concept, not the composed guide.
- **Per-method units.** Frame uses trace correlation, subspace uses ‖B·h‖, so
  `directed_total`/`directed_mean` compare *within* a method; `success_rate` and
  `ppl_ratio` are the cross-method axes.
- **λ dial.** `fluency_weight = λ` blends `concept score + λ · token log-prob`;
  swept over `{0, 0.4}` for both methods (subspace repel via the fluent loop).
- Flip `SMOKE = False` for the full 8-prompt battery. Set `WORD_FIELD = "words"`
  to use the full pooled lists instead of the Borda `core`.